In [ ]:
# Run this cell first!
%pip install -q openai pydantic

from google.colab import userdata, drive
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
drive.mount("/content/drive")

# Build an Encyclopedia from Your Texts

This notebook:
1. **Reads** your texts (from files or pasted directly)
2. **Finds passages** related to a topic you choose
3. **Extracts structured data** for each passage
4. **Generates artwork** with DALL-E
5. **Saves everything** as JSON + images

---

In [ ]:
import json
import requests
from pathlib import Path
from pydantic import BaseModel
from openai import OpenAI
from IPython.display import Image, display, Markdown

client = OpenAI()

# Where to save outputs
OUTPUT_DIR = Path("/content/drive/MyDrive/encyclopedia")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUTPUT_DIR}")

---

## 1. Configuration

Choose your topic and how you want to provide texts.

In [ ]:
# What topic are you looking for?
TOPIC = "birds"

# How do you want to provide texts?
MODE = "manual"  # "manual" = paste texts below, "folder" = read from Drive folder

# If using folder mode, where are your files?
TEXTS_DIR = Path("/content/drive/MyDrive/texts")

---

## 2. Your Texts

**If MODE = "manual":** Edit the texts below.  
**If MODE = "folder":** Put `.txt` or `.md` files in your TEXTS_DIR and skip this cell.

In [ ]:
# Only used if MODE = "manual"
manual_texts = [
    {
        "filename": "genesis.txt",
        "text": """In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters. And God said, Let there be light: and there was light."""
    },
    {
        "filename": "theogony.txt",
        "text": """First of all, the Void came into being, next broad-bosomed Earth, the solid and eternal home of all, and Eros, the most beautiful of the immortal gods, who in every man and every god softens the sinews and overpowers the prudent purpose of the mind."""
    },
]

---

## 3. Load the Texts

In [ ]:
def read_text_files(folder: Path) -> list[dict]:
    """Read all .txt and .md files from a folder."""
    files = []
    for filepath in sorted(folder.glob("*")):
        if filepath.suffix in [".txt", ".md"]:
            files.append({
                "filename": filepath.name,
                "text": filepath.read_text()
            })
    return files


# Load texts based on mode
if MODE == "folder":
    source_files = read_text_files(TEXTS_DIR)
    print(f"Loaded {len(source_files)} files from {TEXTS_DIR}")
else:
    source_files = manual_texts
    print(f"Using {len(source_files)} manually entered texts")

for f in source_files:
    print(f"  - {f['filename']} ({len(f['text'])} chars)")

---

## 4. Find Relevant Passages

Use GPT to find passages related to your topic.

In [ ]:
class Passage(BaseModel):
    quote: str          # The exact passage from the text
    context: str        # Brief explanation of why it's relevant

class PassageList(BaseModel):
    passages: list[Passage]


def find_passages(text: str, topic: str, source: str) -> list[Passage]:
    """Find passages about a topic in a text."""
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Find all passages in this text that relate to "{topic}".
Extract the exact quotes. If there are no relevant passages, return an empty list.

Text from {source}:

{text}"""
        }],
        response_format=PassageList
    )
    return response.choices[0].message.parsed.passages

In [ ]:
# Find passages in all files
all_passages = []

for file in source_files:
    print(f"Searching {file['filename']}...")
    passages = find_passages(file["text"], TOPIC, file["filename"])
    
    for i, p in enumerate(passages):
        all_passages.append({
            "id": f"{Path(file['filename']).stem}-{i+1}",
            "source": file["filename"],
            "quote": p.quote,
            "context": p.context
        })
    
    print(f"  Found {len(passages)} passages")

print(f"\nTotal: {len(all_passages)} passages about '{TOPIC}'")

---

## 5. Save Passages as Markdown

In [ ]:
# Build markdown content
md_lines = [f"# Passages about {TOPIC}\n"]

for p in all_passages:
    md_lines.append(f"## {p['id']}\n")
    md_lines.append(f"**Source:** {p['source']}\n")
    md_lines.append(f"> {p['quote']}\n")
    md_lines.append(f"*{p['context']}*\n")
    md_lines.append("---\n")

md_content = "\n".join(md_lines)

# Save to file
passages_file = OUTPUT_DIR / f"passages-{TOPIC}.md"
passages_file.write_text(md_content)
print(f"Saved to: {passages_file}")

# Display preview
display(Markdown(md_content[:2000] + "\n\n*[truncated...]*" if len(md_content) > 2000 else md_content))

---

## 6. Define the Encyclopedia Entry Schema

In [ ]:
class Entry(BaseModel):
    title: str
    summary: str
    themes: list[str]
    era: str
    art_prompt: str  # Used to generate the image

---

## 7. The Processing Functions

In [ ]:
def extract_entry(passage: dict) -> Entry:
    """Extract structured data from a passage."""
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Create an encyclopedia entry for this passage.
For art_prompt, describe a vivid scene suitable for illustration.

Source: {passage['source']}
Passage: {passage['quote']}"""
        }],
        response_format=Entry
    )
    return response.choices[0].message.parsed


def generate_image(prompt: str, filename: str) -> str:
    """Generate an image with DALL-E and save it."""
    response = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size="1024x1024",
        quality="standard",
        n=1
    )
    
    img_data = requests.get(response.data[0].url).content
    filepath = OUTPUT_DIR / filename
    filepath.write_bytes(img_data)
    return str(filepath)

---

## 8. Generate the Encyclopedia

In [ ]:
entries = []

for passage in all_passages:
    print(f"Processing {passage['id']}...")
    
    # Extract structured data
    entry = extract_entry(passage)
    
    # Generate artwork
    image_file = f"{passage['id']}.png"
    image_path = generate_image(entry.art_prompt, image_file)
    
    # Build the final entry
    entry_data = entry.model_dump()
    entry_data["id"] = passage["id"]
    entry_data["source"] = passage["source"]
    entry_data["quote"] = passage["quote"]
    entry_data["image"] = image_file
    entries.append(entry_data)
    
    # Show result
    print(f"  -> {entry.title}")
    display(Image(filename=image_path, width=300))
    print()

print(f"Done! Created {len(entries)} entries.")

---

## 9. Save the Data

In [ ]:
output_file = OUTPUT_DIR / f"entries-{TOPIC}.json"

with open(output_file, "w") as f:
    json.dump(entries, f, indent=2)

print(f"Saved to: {output_file}")
print()
print(json.dumps(entries[0], indent=2) if entries else "No entries created.")

---

## Done!

Your files are in Google Drive:
- `encyclopedia/passages-{topic}.md` — extracted passages
- `encyclopedia/entries-{topic}.json` — structured data
- `encyclopedia/*.png` — generated images

**To run again with different settings:**
1. Change `TOPIC` to search for something else
2. Switch `MODE` between "manual" and "folder"
3. Add more texts (paste them or add files to your folder)
4. Run all cells again